# # Dataset Summary
It counts the number of rows, drives, date range and rows which have the failures across all 91 daily CSVs. This gave me roughly 32 million rows with 359,101 drives ranging from April 1 to June 30 with only 1,522 failures in the data.

In [5]:
import duckdb
from pathlib import Path

DATA_GLOB = str(Path("../data/raw/*.csv"))
con = duckdb.connect()

summary = con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT serial_number)     AS drives,
        MIN(date)                         AS first_day,
        MAX(date)                         AS last_day,
        SUM(CAST(failure AS INTEGER))     AS failure_rows
    FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
""").df()
summary

,total_rows,drives,first_day,last_day,failure_rows
0,31968003,359101,2026-04-01,2026-06-30,1522.0


# Slimming the file and failures in each model
The file size of the CSVs is 12GB which can be very time consuming. What we are doing here is that, we are copying 5 important columns into a compact file format known as Parquet. Parquet is similar to photocopying only the key columns of every page in order for us to flip through the pages faster. It then counts the failures we get for each model. The model HGST HUH721212ALN604 failes the most as it failed 180 times followed by OSHIBA MG07ACA14TA with 179 failures in the span of Q2. Whats important to notice is that the percantge of faialures based on the number of drives over the failures.

In [6]:
con.sql(f"""
    COPY (
        SELECT
            CAST(date AS DATE)                 AS date,
            serial_number,
            model,
            TRY_CAST(capacity_bytes AS BIGINT) AS capacity_bytes,
            CAST(failure AS INTEGER)           AS failure
        FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
    ) TO '../data/drive_index.parquet' (FORMAT PARQUET)
""")

models = con.sql("""
    SELECT model,
           COUNT(DISTINCT serial_number)  AS drives,
           SUM(failure)                   AS failures,
           ROUND(100.0 * SUM(failure) / COUNT(DISTINCT serial_number), 2) AS pct_failed
    FROM '../data/drive_index.parquet'
    GROUP BY model
    HAVING SUM(failure) > 0
    ORDER BY failures DESC
    LIMIT 15
""").df()
models

,model,drives,failures,pct_failed
0,HGST HUH721212ALN604,9694,180.0,1.86
1,TOSHIBA MG07ACA14TA,37171,179.0,0.48
2,ST12000NM0008,18521,145.0,0.78
3,ST8000NM0055,13112,112.0,0.85
4,HGST HUH721212ALE604,13196,110.0,0.83
5,TOSHIBA MG08ACA16TA,39932,102.0,0.26
6,ST24000NM002H,9631,82.0,0.85
7,WDC WUH721816ALE6L4,26286,74.0,0.28
8,TOSHIBA MG10ACA20TE,25270,64.0,0.25
9,WDC WUH722222ALE6L4,45665,63.0,0.14


# SMART columns available
Every drive reports SMART values such as error counts and temnperature. Different drives have different readings. A good way to understand would be that different car brands have a different dashboards. The following shows how often each reading is filled. The data is normalised between 0 and 100 where 0 means its never filled and 100 means that its always filled in

In [7]:
cands = ["HGST HUH721212ALN604", "TOSHIBA MG07ACA14TA", "ST12000NM0008"]
in_list = ", ".join(f"'{m}'" for m in cands)

fill = con.sql(f"""
    SELECT model, COUNT(*) AS n_rows, COUNT(COLUMNS('^smart_.*_raw$'))
    FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
    WHERE model IN ({in_list})
    GROUP BY model
""").df().set_index("model")

pct = (fill.drop(columns="n_rows").div(fill["n_rows"], axis=0) * 100).round(0)
pct.columns = [c.replace("count(", "").replace(")", "") for c in pct.columns]
pct.T[(pct.T > 0).any(axis=1)]

model,HGST HUH721212ALN604,ST12000NM0008,TOSHIBA MG07ACA14TA
smart_1_raw,100.0,100.0,100.0
smart_2_raw,100.0,0.0,100.0
smart_3_raw,100.0,100.0,100.0
smart_4_raw,100.0,100.0,100.0
smart_5_raw,100.0,100.0,100.0
smart_7_raw,100.0,100.0,100.0
smart_8_raw,100.0,0.0,100.0
smart_9_raw,100.0,100.0,100.0
smart_10_raw,100.0,100.0,100.0
smart_12_raw,100.0,100.0,100.0


# The sampled dataset
This is to keep all the failed drives as they are the rare ones therefore adding the real value which we are after. We know which 3 models we need to use from above. These 3 still have 65,000 drives with millions of rows with most being healthy. We are trying to keep 20 in every 100 healthy drives  where it keeps the basic columns and the SMART columns which are filled for atleast 1 of the 3 models. Each drive is prresented with2 chances to be kept. The first one is if it is a failed drive then it is kept and the second way is if it passes the 20% rule which turns each serial number in a range of 0 to 99. If the number is below 20 then we keep the drive else we get rid of it. We do keep the whole drive instead of just a few realtive rows. The reason behind that is that we have a model that fails on day 56 but till day 55 it would show up as healthy so for us to train the mdoel that history would be really important. The output is then sorted by their serial numbers and dates so the history shows up in order. I perform validation by just running it and counting total number of failures and then verified by cross checking from above to make sure we didn't lose and failed drive whilst the process.

In [8]:
HEALTHY_PCT = 20

raw_cols = list(pct.columns[(pct > 0).any(axis=0)])
norm_cols = [c.replace("_raw", "_normalized") for c in raw_cols]
smart_sql = ",\n            ".join(f"TRY_CAST({c} AS DOUBLE) AS {c}" for c in raw_cols + norm_cols)

con.sql(f"""
    COPY (
        SELECT
            CAST(date AS DATE)                 AS date,
            serial_number,
            model,
            TRY_CAST(capacity_bytes AS BIGINT) AS capacity_bytes,
            CAST(failure AS INTEGER)           AS failure,
            {smart_sql}
        FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
        WHERE model IN ({in_list})
          AND (serial_number IN (SELECT serial_number
                                 FROM '../data/drive_index.parquet'
                                 WHERE failure = 1)
               OR hash(serial_number) % 100 < {HEALTHY_PCT})
        ORDER BY serial_number, date
    ) TO '../data/sampled.parquet' (FORMAT PARQUET)
""")

con.sql("""
    SELECT model,
           COUNT(*)                       AS rows,
           COUNT(DISTINCT serial_number)  AS drives,
           SUM(failure)                   AS failures
    FROM '../data/sampled.parquet'
    GROUP BY model
""").df()

,model,rows,drives,failures
0,HGST HUH721212ALN604,179438,2118,180.0
1,ST12000NM0008,337854,3815,145.0
2,TOSHIBA MG07ACA14TA,679085,7578,179.0
